# 🌿 CarbonWise — Carbon Emission Prediction
### Linear Regression Model | College Project
---

## 1. Setup & Data Loading

In [ ]:
!git clone https://github.com/Dakshgupta25/CarbonWise.git
%cd CarbonWise

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set consistent plot style throughout the notebook
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Load the dataset
df = pd.read_excel("data/Carbon Emission.xlsx")

# Preview the first few rows to understand structure
df.head()

**Understanding:** We import all required libraries upfront and load the raw dataset. Setting a consistent plot theme ensures all visualizations look uniform throughout the notebook.

---
## 2. Data Cleaning

In [ ]:
# Basic shape and type overview
print(f"Shape: {df.shape}")
print(f"\nDuplicated rows: {df.duplicated().sum()}")
print("\nData Types:")
print(df.dtypes)

**Understanding:** We check the dataset dimensions, look for duplicate rows, and review each column's data type to know which columns are categorical vs numerical before any further processing.

In [ ]:
# Standardise column names: replace spaces with underscores
df.columns = df.columns.str.replace(' ', '_')

# Separate categorical and numerical columns for targeted treatment
cat_col = df.columns[df.dtypes == object]
num_col = df.columns[df.dtypes != object]

print("Categorical columns:", list(cat_col))
print("Numerical columns :", list(num_col))

**Understanding:** Column names are cleaned to remove spaces (makes them easier to reference in code). Columns are then split into categorical and numerical groups so we can apply appropriate cleaning steps to each group separately.

In [ ]:
# Check missing values in both column groups
print("Missing values — Numerical:")
print(df[num_col].isnull().sum())

print("\nMissing values — Categorical:")
print(df[cat_col].isnull().sum())

**Understanding:** We identify where null values exist. This guides our imputation strategy — different column types need different handling (e.g., logical fill based on related columns for categorical, vs statistical fill for numerical).

In [ ]:
# For rows where transport mode is public or walk/bicycle,
# a missing Vehicle_Type logically means 'No Vehicle' — fill accordingly
df['Vehicle_Type'] = df.apply(
    lambda row: 'No Vehicle'
    if pd.isnull(row['Vehicle_Type']) and row['Transport'] in ['public', 'walk/bicycle']
    else row['Vehicle_Type'],
    axis=1
)

print("Remaining nulls in categorical columns:")
print(df[cat_col].isnull().sum())

**Understanding:** Instead of blindly dropping or guessing, we use domain logic — if someone uses public transport or walks, they have no personal vehicle. This is a context-aware imputation that preserves data integrity.

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of all categorical features using count plots
cols = 3
rows = (len(cat_col) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(cat_col):
    sns.countplot(x=df[col], ax=axes[i], order=df[col].value_counts().index)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=40)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Categorical Feature Distributions", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Understanding:** Count plots show how many records fall into each category. This reveals class imbalances (e.g., one vehicle type dominating) and gives us a feel for the data composition before modelling.

In [ ]:
# Identify numerical feature columns (excluding target)
target = 'CarbonEmission'
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols.remove(target)

# Summary statistics for numerical features
stats_df = pd.DataFrame({
    'Mean'    : df[num_cols].mean().round(2),
    'Median'  : df[num_cols].median().round(2),
    'Std Dev' : df[num_cols].std().round(2),
    'Skewness': df[num_cols].skew().round(2),
    'Min'     : df[num_cols].min(),
    'Max'     : df[num_cols].max()
})

print("Numerical Feature Statistics:")
stats_df

**Understanding:** Descriptive statistics summarise the spread and shape of each numerical feature. High skewness values (>1 or <-1) indicate a non-normal distribution that may need transformation. Mean vs median differences also signal skew and outliers.

In [ ]:
# Distribution plots (histograms + KDE) for numerical features
cols = 3
rows = (len(num_cols) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f"{col}  (skew={df[col].skew():.2f})", fontsize=11)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Numerical Feature Distributions", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Understanding:** Histograms with KDE (Kernel Density Estimate) overlay show the actual shape of each distribution. The skewness value in the title confirms visually what the statistics showed — helping us decide which features need log transformation.

In [ ]:
# Correlation heatmap — shows linear relationships between numerical features and target
plt.figure(figsize=(10, 7))
corr_cols = num_cols + [target]
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))  # only lower triangle
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap='coolwarm', vmin=-1, vmax=1,
    square=True, linewidths=0.5
)
plt.title("Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Understanding:** The heatmap shows pairwise linear correlation coefficients. Values close to +1 or -1 mean strong linear relationships. We focus on the last row/column (CarbonEmission) to see which features are most predictive for our target variable.

In [ ]:
# Scatter plots: each numerical feature vs the target (CarbonEmission)
cols = 3
rows = (len(num_cols) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].scatter(df[col], df[target], alpha=0.3, s=10, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('CarbonEmission')
    axes[i].set_title(f"{col} vs CarbonEmission", fontsize=11)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Feature vs Target (CarbonEmission)", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Understanding:** Scatter plots reveal the nature of the relationship between each predictor and the target. A clear upward/downward trend suggests a useful linear predictor. Scattered clouds suggest weak or non-linear relationships.

In [ ]:
# Box plots: how CarbonEmission varies across categories
cat_cols = df.select_dtypes(include=['object']).columns
cols = 3
rows = (len(cat_cols) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.boxplot(x=col, y=target, data=df, ax=axes[i])
    axes[i].set_title(f"{col} vs CarbonEmission", fontsize=11)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=40)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Categorical Features vs CarbonEmission", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Understanding:** Box plots compare the distribution of CarbonEmission across each category. Large differences in median values between categories indicate that feature is a meaningful predictor. Overlapping boxes suggest less discriminative power.

---
## 4. Outlier Treatment & Feature Transformation

In [ ]:
# Detect outliers using IQR method before treatment
outlier_info = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    count = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    outlier_info.append([col, count, round((count / len(df)) * 100, 2)])

outlier_df = pd.DataFrame(outlier_info, columns=['Column', 'Outliers', 'Percent (%)'])
print(outlier_df)

**Understanding:** The IQR (Interquartile Range) method flags values that fall more than 1.5×IQR below Q1 or above Q3 as outliers. Knowing how many outliers exist per column helps us decide the right treatment strategy.

In [ ]:
# Apply log transformation for heavily skewed columns,
# then cap remaining outliers using IQR clipping
df_clean = df.copy()
skewed_cols = df[num_cols].skew().abs()

for col in num_cols:
    # Log-transform features with high skew (abs skew > 1)
    if skewed_cols[col] > 1:
        df_clean[col] = np.log1p(df_clean[col])

    # Cap extreme values using IQR bounds
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    df_clean[col] = np.clip(df_clean[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# Confirm outliers have been eliminated
print("Remaining outliers after treatment:")
for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    count = ((df_clean[col] < Q1 - 1.5 * IQR) | (df_clean[col] > Q3 + 1.5 * IQR)).sum()
    print(f"  {col}: {count}")

**Understanding:** Two-step treatment — (1) log1p transformation compresses heavily skewed distributions towards normality, which is important for linear regression assumptions; (2) IQR capping clips any remaining extremes without losing rows. After treatment, all counts should be 0.

In [ ]:
# Side-by-side boxplots: Raw vs Cleaned for each numerical feature
fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 3.5 * len(num_cols)))

for i, col in enumerate(num_cols):
    axes[i, 0].boxplot(df[col].dropna())
    axes[i, 0].set_title(f"{col} — Raw", fontsize=10)

    axes[i, 1].boxplot(df_clean[col].dropna())
    axes[i, 1].set_title(f"{col} — Cleaned", fontsize=10)

plt.suptitle("Outlier Treatment: Raw vs Cleaned", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Understanding:** Visual confirmation that outlier treatment worked. Comparing raw vs cleaned boxplots shows the reduction in extreme whiskers and outlier points — validating that our log + IQR capping approach effectively cleaned the data.

---
## 5. Feature Engineering & Preprocessing

In [ ]:
# Drop features with low predictive value based on EDA insights
drop_cols = ['Diet', 'Social_Activity', 'Monthly_Grocery_Bill', 'How_Often_Shower']
df_model = df_clean.drop(columns=drop_cols)

# Consolidate rare categories in Cooking_With to avoid excessive dummy columns
top_categories = df_model['Cooking_With'].value_counts().nlargest(5).index
df_model['Cooking_With'] = df_model['Cooking_With'].apply(
    lambda x: x if x in top_categories else 'Other'
)

print("Remaining columns:", df_model.shape[1])
print("Cooking_With categories:", df_model['Cooking_With'].unique())

**Understanding:** We remove features that showed weak or no relationship with CarbonEmission during EDA (low correlation, similar box plot distributions). Rare categories in Cooking_With are grouped as 'Other' to prevent the model from learning noise from tiny categories.

In [ ]:
# Separate features (X) and target (y)
X = df_model.drop(target, axis=1)
y = df_model[target]

# 80-20 train-test split with fixed random seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

**Understanding:** We split the data before any encoding or scaling to prevent data leakage — the test set must simulate completely unseen data. 80% for training gives the model enough data to learn, while 20% for testing gives a reliable performance estimate.

In [ ]:
# One-hot encode categorical columns (drop_first avoids dummy variable trap)
X_train = pd.get_dummies(X_train, drop_first=True)
X_test  = pd.get_dummies(X_test,  drop_first=True)

# Align test set columns to training set (fill any missing with 0)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Remove highly correlated features (r > 0.9) to reduce multicollinearity
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
X_train = X_train.drop(columns=to_drop)
X_test  = X_test.drop(columns=to_drop)

# Scale features — Linear Regression is sensitive to feature magnitudes
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)         # transform test using train stats

print(f"Final feature count: {X_train.shape[1]}")
print(f"Dropped for high correlation: {to_drop}")

**Understanding:** Three important steps here — (1) One-hot encoding converts categorical text into numeric dummy variables that linear regression can process; (2) Dropping highly correlated features (>0.9) removes multicollinearity which inflates coefficient errors in linear regression; (3) StandardScaler normalises all features to zero mean and unit variance so no single feature dominates due to its scale.

---
## 6. Linear Regression — Model Training

In [ ]:
# Train the Linear Regression model on scaled training data
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Generate predictions on the unseen test set
y_pred = lr_model.predict(X_test_scaled)

print("Model training complete.")
print(f"Intercept : {lr_model.intercept_:.4f}")
print(f"No. of coefficients: {len(lr_model.coef_)}")

**Understanding:** Linear Regression fits a hyperplane through the training data by minimising the sum of squared errors. The intercept is the baseline prediction when all features are zero. Each coefficient represents how much CarbonEmission changes per unit change in that feature (after scaling).

---
## 7. Model Evaluation

In [ ]:
# Compute evaluation metrics on the test set
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("═" * 35)
print("   Linear Regression — Results")
print("═" * 35)
print(f"  MAE      : {mae:.2f}")
print(f"  MSE      : {mse:.2f}")
print(f"  RMSE     : {rmse:.2f}")
print(f"  R² Score : {r2:.4f}")
print("═" * 35)

**Understanding:** Four complementary metrics evaluate performance — MAE (average absolute error, easy to interpret in original units), RMSE (penalises large errors more heavily), and R² (proportion of variance in CarbonEmission explained by our model — closer to 1.0 is better).

In [ ]:
# Actual vs Predicted scatter plot — ideal predictions lie on the diagonal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, s=15, color='steelblue', label='Predictions')
min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Carbon Emission')
axes[0].set_ylabel('Predicted Carbon Emission')
axes[0].set_title('Actual vs Predicted', fontsize=13, fontweight='bold')
axes[0].legend()

# Plot 2: Residual distribution
residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1], color='coral', bins=40)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Residuals (Actual − Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

**Understanding:** Two diagnostic plots — (1) Actual vs Predicted: points close to the red diagonal mean the model predicts well; systematic deviations reveal bias; (2) Residual Distribution: for a well-fitted linear model, residuals should be roughly normally distributed and centred at zero. A skewed or bimodal residual plot signals model issues.

In [ ]:
# Top 10 most influential features by absolute coefficient magnitude
coef_df = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Coefficient': lr_model.coef_
})
coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
top10 = coef_df.nlargest(10, 'Abs_Coef')

plt.figure(figsize=(10, 5))
colors = ['steelblue' if c > 0 else 'coral' for c in top10['Coefficient']]
plt.barh(top10['Feature'], top10['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title('Top 10 Feature Coefficients — Linear Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Understanding:** Feature coefficients from Linear Regression directly tell us each feature's impact on the predicted CarbonEmission. Positive coefficients (blue) increase emission; negative (coral) decrease it. Larger absolute values indicate more influential features — this is the linear model's built-in interpretability advantage.

---
## 8. Save the Model

In [ ]:
import pickle

# Save the trained model and scaler together so predictions can be reproduced
model_bundle = {
    'model'   : lr_model,
    'scaler'  : scaler,
    'features': list(X_train.columns)
}

with open('carbon_lr_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

print("Model saved as 'carbon_lr_model.pkl'")
print(f"Features used ({len(X_train.columns)}): {list(X_train.columns)}")

**Understanding:** We save the model, scaler, and feature list together in one bundle. This is important because any new prediction must go through the same scaler (fitted on training data) and use the exact same feature columns in the same order — saving them together prevents mismatches during deployment.

---
## Summary

| Step | Action |
|------|--------|
| Data Loading | Loaded Carbon Emission dataset from Excel |
| Data Cleaning | Fixed missing values, standardised column names |
| EDA | Count plots, histograms, correlation heatmap, scatter & box plots |
| Outlier Treatment | IQR detection → log transform + clipping |
| Feature Engineering | Dropped low-value features, one-hot encoding, removed high-correlation features |
| Scaling | StandardScaler applied (fit on train, transform on test) |
| Model | Linear Regression trained on 80% of data |
| Evaluation | MAE, MSE, RMSE, R² + Actual vs Predicted + Residual plots |
| Saving | Model + scaler + feature list saved as `.pkl` |